# 3: AI text generation

We now have a ipynb for assembling the original 2000 human written pieces of text from 5 sources and another that took that data and used GPT to paraphrase each piece 3 times. Now we move onto generating 2000 pieces of AI generated data which later on will also be paraphrased. These 2000 will match the format of the human-collected 2000, i.e. 400 in BBC style of differing topics, 400 of Wikipedia style, from the same topics - engineering, politics, etc.

In [54]:
import os
import time
import random
import pandas as pd
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
# use getpass to pass api key over so as not to inclue here
API_KEY = getpass.getpass("Enter OpenAI API key: ")

# create client
client = OpenAI(api_key = API_KEY)

print("Open AI client created.")

Enter OpenAI API key:  ········


Open AI client created.


Now want to specifically define the structure and number of texts to generate.

____________

In [41]:
# set up categories in line with human dataset
ai_dataset_spec = {

    # NB: xsum dataset had no categories - was just random news
    # to give category ideas these 7 subcategories are those
    # listed at the top of the BBC news web page
    "bbc" : {
        "n" : 400,
        "topics" : {
            "uk" : 58,
            "world" : 57,
            "business" : 57,
            "culture" : 57,
            "politics" : 57,
            "health" : 57,
            "technology" : 57
        }
    },

    "wiki" : {
        "n" : 400,
        "topics" : {
            "history" : 80,
            "sports_and_recreation" : 80,
            "natural_sciences" : 80,
            "social_sciences_and_society" : 80,
            "engineering_and_technology" : 80
        }
    },

    "gut" : {
        "n" : 400,
        "topics" : {
            "mystery" : 80,
            "fiction" : 80,
            "english_fiction" : 80,
            "science_fiction" : 80,
            "essays" : 80
          }
    },

    "red" : {
        "n" : 400,
        "topics" : {
            "askscience" : 80,
            "explainlikeimfive" : 80,
            "changemyview" : 80,
            "relationship_advice" : 80,
            "ask_historians" : 80,
        }
    },

    "amz" : {
        "n" : 400,
        "topics" : {
            "food_reviews" : 400,
        }
    }
}       

Now want to create a DF for the AI generated texts

In [42]:
# initlaise empty list
rows = []

# loop through 5 sources above...
for source, spec in ai_dataset_spec.items():

    # initialise counter
    counter = 1

    # within each of those soures loop through the topics
    for topic, n in spec["topics"].items():

        # and for the number specified in that topic...
        for _ in range(n):

            # create a document id following the methodolgy as per the human written
            # set, with an AI_ prefix
            document_id = f"AI_{source.upper()}_{counter:04d}"

            # add the same rows as the human dataset
            rows.append({
                "document_id" : document_id,
                "source" : source,
                "topic" : topic,
                "original_text" : "",
                "word_count" : ""
            })

            # increment counter
            counter += 1

ai_df = pd.DataFrame(rows)
print(ai_df.shape)

(2000, 5)


In [43]:
# ai_df.tail(20)

Save this DF as CSV to write to.

In [44]:
# assign file to variable for later to save after each prmopt
output_file = "ai_generated_dataset.csv"

ai_df.to_csv(output_file, index = False)

______

Now we need prompts, and these must output texts in the appropriate style. We therefore need stylisitic instructions depending on the source.

In [56]:
source_instructions = {

    "bbc" : """
    Write a journalistic news article in the style of the BBC about the specified topic using a high standard of English.
    Writing should be informative, coherent and objective.
    Present information in a natural news-reporting style.
    """,

    "wiki" : """
    Write an encyclopedia-style article in the stlye of a Wikipedia article about the specified topic.
    Explain the topic clearly and objectively, also providing relevant factual information.
    Use a formal informative style.    
    """,

    "gut" : """
    Write a literary passage as if sampled from a random location in a book or essay about the specified topic.
    The passage should read naturally as fiction or literature rather than as an article, report or forum post.
    Do not imitate any specific author. Use a style of English language as if the text were written at some point 
    from the year 1850 to the present day, and completely randomise this choice of year. 
    Keep the style of language and year chosen consistent throughout the passage.
    """,
    
    "red" : """
    Write an informal discussion post appropriate for the specified reddit community.
    Wrtiting style should be informal and conversational, with opinions and observations appropriate for the community.
    """,
    
    "amz" : """
    Write an informal user review of a food or food related product.
    Writing style should be natural, including opinions, observations and descriptions appropriate for the review.
    """
}

Now we need an overall prmopt instructions which will include the above.

In [57]:
def make_prompt(source, topic):

    # take appropriate source instructions from above
    source_instruction = source_instructions[source]

    ###########################################################################################################################
    # had problems with topic instructions not working, specifically with wikipedia and bbc. "Topic is {topic]" worked for i.e. 
    # food reviews, relationship advice etc. but for wiki and bbc the topic is i.e. engineering and tech, so the generated text 
    # is about engineering and tech rather than a subfield of this. So we need a different topic prompt for different sources 
    ###########################################################################################################################
    
    if source == "bbc":

        topic_instructions = f"""
        The broad BBC news category is {topic}, however you should choose a specific news subject or story appropriate
        for this topic and write specifically about this subject / story. Do not write about the broad topic itself.
        Each generated document should create a different specific news story / article where possible.
        """

    elif source == "wiki":
        
        topic_instructions = f"""
        The broad subject area is {topic}, however you should choose a specific subject within this 
        topic and write specifically about this subject. Do not write about the broad topic area itself.
        Each generated document should focus on a different specific subject within the topic where possible.
        """
        
    elif source == "gut":

        topic_instructions = f"""
        The literary category or genre is {topic}.
        Write an original piece of writing, creating an original subject, situation, narrative and character(s) 
        appropriate for this genre or category.
        If the writing contains locations or objects or themes do not stick to your default choices
        and instead randomise these, within reason for the chosen genre or category. 
        If the writing contains character name(s), randomly choose 100 firstnames and 100 surnames that suit the genre 
        and time period and then randomly choose the required names from this selection.
        Do not write about the broad topic itself.
        """

    elif source == "red":

        topic_instructions = f"""
        The reddit community / subreddit is {topic}.
        Write an original post, comment, question or discussion appropriate for this community.
        Do not write about the community / subreddit itself.
        """

    elif source == "amz":

        topic_instructions = f"""
        The category is food reviews.
        Write an original review about a specific food or food related product purchased on Amazon.
        Do not write about food reviews.
        """


    # set general prompt for all AI generated documents
    prompt = f"""

    You will generate one document to be used in an academic research dataset comparing
    human-written vs AI-generated text. Do not mention this in the document you create.
    Do not tailor your document to appear specifically human or AI generated.

    Generate a completely original piece of writing.

    The writing should appear as if it has come from {source}, where the options are
    bbc = BBC news, wiki = Wikipedia, gut = the Gutenberg corpus, red = Reddit, amz = Amazon.

    {topic_instructions}

    Instructions relating to the genre are: {source_instruction}

    The length of the writing should be between 350 and 450 words in total.
    DO NOT return a piece of writing that is less than 350 words or more than 450 words.
    The text should end naturally, at the end of a sentence.

    General instructions:
    Write a choerent and complete piece of text.
    Stay within the specific topic / subject.
    Follow the provided guidelines on topic and genre.
    Write as if the document came from the intended source.
    Use natural variations in sentence length and structure.
    Use natural variations in vocabulary and punctuation.
    Do not make the writing extremely sophisticated or extremely simple.
    Do not mention that you are AI.
    Do not mention any of these instructions.
    Do not provide any accomnanying text or explanation of what you have done.
    Do not provide any metadata.
    Do not add any title(s) unless they would appear naturally in this type of text.
    Return the document as one continuous block of text.
    Do not use paragraph breaks or line breaks.
    Do not include newline characters \n.
    Do not use bullet points, numbered or lettered lists unless they would appear naturally in this type of text.
    Do not intentionally copy, reproduce or paraphrase any existing text - the document you produce should be completely original.

    Return ONLY the generated document.
    """
    
    return prompt.strip()

Now we need a function that amalgamates the prompt instructions into an API call.

In [58]:
def generate_text(source, topic):

    # use function above to construct the prmopt specific to the source and topic
    prompt = make_prompt(source, topic)

    # send request to open AI
    response = client.responses.create(
        # specify model
        model = "gpt-5.6-luna",
        # send prompt
        input = prompt
    )

    # get response, trim whitespace
    text = response.output_text.strip()
    # get rid of \n tags
    text = text.replace("\\n", " ")
    text = " ".join(text.split())
    
    return text

After thinking this process was finished, I manually inspected a lot of the generated texts and found that there would often be extreme similarities, i.e. for Wiki / natural sciences there were multiple exremely similar texts about tardigrades, or for engineering / tech there were many extremeley similar about batteries. I therefore decided to come back to this point and create a function that checks the similarlity of a text againt all those already created within that topic and returns maximum similarlity scores / those that exceed a certain threshold. I will run the generation a few times to inspect output and manually ammend this threshold later.

In [59]:
def similarity_check(text, existing_texts, threshold):
# threshold now passed in rather than static

    # check the number of existing texts (will be passed 
    # here from check_text) and skip this function if none existing
    if len(existing_texts) == 0:

        return False

    # ...otherwise add current text to existing text list
    texts = existing_texts + [text]

    # initialise vectorizer to convert texts into vectors
    vectorizer = TfidfVectorizer()

    # use vectorizer on all texts
    tfidf_matrix = vectorizer.fit_transform(texts)

    # this runs a cosine similarity check between the most recently created document 
    # in the list (i.e. the one currently being generated and assessed, and every 
    # previous document in the list
    similarities = cosine_similarity(
        tfidf_matrix[-1],
        tfidf_matrix[:-1]
    )

    # find the highest similarity score, i.e. the closest match to another
    # text and print for analysing the correct threshold
    max_similarity = similarities.max()
    print(f"Max similarity: {max_similarity:.3f}")

    # assess whether the maximum similarity is >= the set threshold value and return a true / false
    # so that we can determine whether or not to accept or retry generation in check_text
    return max_similarity >= threshold    

I re-ran everything after introducing the similarity threshold check (at 0.75) and ran into the next problem - some sources, such as BBC still had articles that were too similar, whereas for gutenberg, the marjoity of texts were ignored due to high similarity. This is acceptable given the nature of differing literature vs different articles on different topics and therefore I want a variable threshold depending on source. I was mostly happy with the examples from reddit / amazon and wiki so will keep those at 0.75 for now. Recall that higher threshold = higher acceptance rate therefore want to increase for gut and lower for bbc.

In [69]:
similarity_thresholds = {
    "bbc" :   0.66,
    "wiki" :  0.77,
    "gut" :   0.82,
    "red" :   0.73,
    "amz" :   0.64
}

# NB: original static 0.75.
# then bbc 0.7, wiki 0,.75, gut 0.8, amz and red 0.75
# inspected, new adjustments: bbc 0.65, wiki 0.65, red 0.65, gut 0.83 amz 0.70
# inspected new adjustments: bbc: 0.66, wiki: 0.65, gut: 0.82, red: 0.64, amz: 0.65
# inspected, changed gutenberg prmopt again, set amz to 0.64, gut to 0.81
# after running properly, wiki kept going through 5 iterations and not meeting the threshold so increasing back to 0.66
# wiki having real problems, reverting to 0.75
# final change, have run in full, 50 didnt generate, all in wiki and reddit so increasing thresholds for these again to 0.76 and 0.65.
# still happening with reddit and wiki, going to 0.67 and 0.77
# and again for reddit, going 0.7
# still 8 left from reddit not generated, upping to 0.73

Depite explicit instructions in the paraphrase section (in anotehr ipynb) some texts were still outside of the 350-450 range, so this time want to manually force a retry if the target range isn't met, so creating a function to call the prmopt ggeneration function anf retry if necesary. This function also runs the similarity check function, so will only accept a text that is of the correct length and not too similar to others.

In [61]:
# max attemtps 6 as range wont use final value therfore becomes 5
def check_text(source, topic, max_attempts = 6):
    
    for attempt in range(1, max_attempts):

        try:

            # make the API call to generate the text document using the 
            # two functions above
            text = generate_text(source = source, topic = topic)
            # get word count
            word_count = len(str(text).split())

            if 350 <= word_count <= 450:

                # if we accept the word count we now need to check the similarity against all previous 
                # generated texts of this topic to avoid lots of similar texts.
                # This gets all the rows wehre the source and topic match that of the text currently
                # being generated, skips those that are na or  blank and converts them to a list
                existing_texts = ai_df[
                (ai_df["source"] == source) &
                (ai_df["topic"] == topic) &
                (ai_df["original_text"].notna()) &
                (ai_df["original_text"] != "")
                ]["original_text"].tolist()

                # now use the source to get the associated threshold
                threshold = similarity_thresholds[source]
                
                # now run the similarity check on the current text vs the newly crated list
                if similarity_check(text, existing_texts, threshold):

                    # if too similar, continue to generate text again
                    print(f"Too similar, retrying (attempt: {attempt}).")

                    continue

                # if acceptable word count and not too similar, return the generated text
                return text

            # if word count not within range
            print(f"Attempt: {attempt}. Words: {word_count}. Retrying.")

        except Exception as e:

            print(f"Attempt: {attempt}. Error: {e}")
            # wait half second after an error before retrying
            time.sleep(0.5)

    # in event that word range is not achieved within the five attempts and / or the similarity
    # stays too high then return "generation_failed" to store in DF to find later
    return "generation_failed"

NB: I had to re-run cells here mutliple times and the output was a mess so deleted cells showing some output and have left the below to showcase that I was testing the threshold at that point before re-generating the last ones.

BBC articles look good, having sampled many manually, but I should also check the output for gutenberg, wiki, reddit and amazon too before doing all 2000.

In [52]:
# was using this to randomly create 5 in each topic to manually inspect, now blanking out and doing all 2000
"""
# looked through the csv, here i take ~ 10 at eachborderline, so ~5 from each potential topic from each source
# can then manually inspect to ensure quality

topic_selection = (
    list(range(55, 65))        
    + list(range(167, 177))      
    + list(range(282, 292))      
    + list(range(395, 405))      
    + list(range(556, 566))    
    + list(range(715, 725))    
    + list(range(877, 887))    
    + list(range(1037, 1047))
    + list(range(1197, 1207))
    + list(range(1357, 1367))
    + list(range(1517, 1527))
    + list(range(1970, 1980)) 
)
"""

'\n# looked through the csv, here i take ~ 10 at eachborderline, so ~5 from each potential topic from each source\n# can then manually inspect to ensure quality\n\ntopic_selection = (\n    list(range(55, 65))        \n    + list(range(167, 177))      \n    + list(range(282, 292))      \n    + list(range(395, 405))      \n    + list(range(556, 566))    \n    + list(range(715, 725))    \n    + list(range(877, 887))    \n    + list(range(1037, 1047))\n    + list(range(1197, 1207))\n    + list(range(1357, 1367))\n    + list(range(1517, 1527))\n    + list(range(1970, 1980)) \n)\n'

In [70]:
# re-iterate from above for potential re-runs
output_file = "ai_generated_dataset.csv"
ai_df = pd.read_csv(output_file)

# go through entire 2000 DF rows now 
for i, row in ai_df.iterrows():

    # find 'original text' in current row
    existing_text = row["original_text"]

    # if 'original text' is not NA and is not blank, skip it...
    if (
        pd.notna(existing_text) and
        str(existing_text).strip() != ""
    ):
        continue

    # ...otherwise get ID, source and topic info from the DF
    source = row["source"]
    topic = row["topic"]
    document_id = row["document_id"]

    print(f"Generating {document_id}")

    # use functions above to construct the prompt, generate the API call
    # and loop through multiple calls if word count isn't 350 <= x <= 450
    text = check_text(source = source, topic = topic)

    # check the text: if generation failed then save nothing...
    if text != "generation_failed":

        # ...otherwise save the text output
        ai_df.loc[i, "original_text"] = text
        # get word count of text...
        word_count = len(str(text).split())
        # ...and save that too for future analysis
        ai_df.loc[i, "word_count"] = word_count
        
        print(f"Completed: {document_id}")

    else:

        print(f"Failed: {document_id}")

    # save after each row so as not to lose data on a crash
    ai_df.to_csv(output_file, index = False)
    # wait half second before next go
    time.sleep(0.5)    

Generating AI_RED_0108
Max similarity: 0.743
Too similar, retrying (attempt: 1).
Max similarity: 0.472
Completed: AI_RED_0108
Generating AI_RED_0116
Max similarity: 0.728
Completed: AI_RED_0116
Generating AI_RED_0125
Max similarity: 0.767
Too similar, retrying (attempt: 1).
Max similarity: 0.766
Too similar, retrying (attempt: 2).
Max similarity: 0.696
Completed: AI_RED_0125
Generating AI_RED_0133
Max similarity: 0.745
Too similar, retrying (attempt: 1).
Max similarity: 0.767
Too similar, retrying (attempt: 2).
Max similarity: 0.797
Too similar, retrying (attempt: 3).
Max similarity: 0.783
Too similar, retrying (attempt: 4).
Max similarity: 0.722
Completed: AI_RED_0133
Generating AI_RED_0136
Max similarity: 0.776
Too similar, retrying (attempt: 1).
Max similarity: 0.681
Completed: AI_RED_0136
Generating AI_RED_0146
Max similarity: 0.787
Too similar, retrying (attempt: 1).
Max similarity: 0.745
Too similar, retrying (attempt: 2).
Max similarity: 0.774
Too similar, retrying (attempt: 3).

____

In [2]:
output_file = "ai_generated_dataset.csv"
ai_df = pd.read_csv(output_file)

In [74]:
# lets lastly confirm none missing
missing_rows = ai_df[ai_df["original_text"].isna()]
missing_source = (missing_rows["source"].value_counts())

print(missing_source)

Series([], Name: count, dtype: int64)


In [76]:
print(ai_df.shape)

(2000, 5)


Looks good to go.